
# Qwen2.5-3B Abstract Evaluator SFT (Unsloth QLoRA)

This notebook fine-tunes `Qwen/Qwen3-4B` for abstract-quality scoring + rationale generation using chat-format SFT data.

Target mapping:

- **Input**: `Task + Reference + Rubric + Submission`
- **Output**: JSON string with `score` and `rationale`

Designed for **RTX 4060 8GB VRAM** with memory-efficient Unsloth QLoRA.


In [7]:
import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

2.5.1+cu121
12.1
True
NVIDIA GeForce RTX 4060 Laptop GPU


In [1]:

## If needed, uncomment and run once in your environment.
!pip install -U "unsloth[colab-new]" datasets transformers trl accelerate evaluate rouge_score sacrebleu bert-score wandb scikit-learn pandas numpy


  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
  Using cached torchvision-0.27.0-cp310-cp310-win_amd64.whl.metadata (5.5 kB)
   ---------------------------------------- 0.0/71.1 MB ? eta -:--:--
    --------------------------------------- 1.0/71.1 MB 6.3 MB/s eta 0:00:12
   - -------------------------------------- 3.4/71.1 MB 9.1 MB/s eta 0:00:08
   -- ------------------------------------- 5.2/71.1 MB 9.1 MB/s eta 0:00:08
   ---- ----------------------------------- 7.6/71.1 MB 9.6 MB/s eta 0:00:07
   ----- ---------------------------------- 10.5/71.1 MB 10.4 MB/s

In [8]:
import os
import re
import ast
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

PROJECT_ROOT = Path(r"C:\Users\hanib\Desktop\nlp_final_final_final_final_project")
DATA_DIR = PROJECT_ROOT / "Abstract-Evaluator" / "data" / "engsaf" / "clean"

TRAIN_DIR = DATA_DIR / "train"
VAL_DIR   = DATA_DIR / "validation"
TEST_DIR  = DATA_DIR / "test"

# Gather all CSVs from each split folder
train_csvs = sorted(TRAIN_DIR.glob("*.csv"))
val_csvs   = sorted(VAL_DIR.glob("*.csv"))
test_csvs  = sorted(TEST_DIR.glob("*.csv"))

print("Project root:", PROJECT_ROOT)
print("Data dir:", DATA_DIR)
print("Train CSVs:", [p.name for p in train_csvs])
print("Validation CSVs:", [p.name for p in val_csvs])
print("Test CSVs:", [p.name for p in test_csvs])

# Example: load and combine each split
train_df = pd.concat([pd.read_csv(f) for f in train_csvs], ignore_index=True) if train_csvs else pd.DataFrame()
val_df   = pd.concat([pd.read_csv(f) for f in val_csvs], ignore_index=True) if val_csvs else pd.DataFrame()
test_df  = pd.concat([pd.read_csv(f) for f in test_csvs], ignore_index=True) if test_csvs else pd.DataFrame()

print("Shapes:", train_df.shape, val_df.shape, test_df.shape)

Project root: C:\Users\hanib\Desktop\nlp_final_final_final_final_project
Data dir: C:\Users\hanib\Desktop\nlp_final_final_final_final_project\Abstract-Evaluator\data\engsaf\clean
Train CSVs: ['1184_entries.csv', '2368_entries.csv', '3552_entries.csv', '4735_entries.csv']
Validation CSVs: ['338_entries.csv']
Test CSVs: ['506_entries.csv']
Shapes: (11839, 6) (338, 6) (506, 6)



## Dataset Utilities

This block supports two modes:

1. **Preferred**: use existing `train.jsonl / dev.jsonl / test.jsonl` with `messages`.
2. **Fallback**: build a training set from OpenReview CSV with simple weak labels, then split.

If you already generated your high-quality synthetic dataset, place the JSONL files in `data/` and skip fallback quality concerns.


In [9]:
from datasets import Dataset, DatasetDict

def load_or_build_splits() -> DatasetDict:
    # check loaded DataFrames
    missing = []
    if train_df.empty:
        missing.append("train")
    if val_df.empty:
        missing.append("validation")
    if test_df.empty:
        missing.append("test")

    if missing:
        raise FileNotFoundError(
            "Prepared DataFrame splits are required. Missing: " + ", ".join(missing)
        )

    return DatasetDict({
        "train": Dataset.from_pandas(train_df, preserve_index=False),
        "validation": Dataset.from_pandas(val_df, preserve_index=False),
        "test": Dataset.from_pandas(test_df, preserve_index=False),
    })

datasets_dict = load_or_build_splits()
print(datasets_dict)

print("Train sample:")
print(datasets_dict["train"][0])  # show full first row

DatasetDict({
    train: Dataset({
        features: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale'],
        num_rows: 11839
    })
    validation: Dataset({
        features: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale'],
        num_rows: 338
    })
    test: Dataset({
        features: ['question', 'student_answer', 'reference_answer', 'mark_scheme', 'score', 'rationale'],
        num_rows: 506
    })
})
Train sample:
{'question': 'What is an orthotropic material ?', 'student_answer': 'Material in which there is symmetry about 2 orthogonal planes.\n', 'reference_answer': 'Orthotropic materials have 9 elastic constants.They also have material properties at a particular point which differ along 3 orthogonal axes.', 'mark_scheme': "{'0': 'Incorrect response', '1': 'Partially correct response', '2': 'Correct response'}", 'score': 0, 'rationale': '  The definition provided only highlights a property of som


## Load Model (Unsloth QLoRA)

Why no manual 50-70% freezing?

- QLoRA already freezes base weights and trains only LoRA adapters.
- This is more memory-efficient and typically better than manually freezing large contiguous layer blocks for this setup.
- We control trainable capacity via LoRA rank/alpha/target modules.


In [12]:
!python -m pip uninstall -y unsloth unsloth-zoo transformers torchao tokenizers
!python -m pip cache purge

# remove leftover transformers folders manually (important)
!rmdir /s /q C:\Users\hanib\AppData\Local\Programs\Python\Python310\Lib\site-packages\transformers
!rmdir /s /q C:\Users\hanib\AppData\Local\Programs\Python\Python310\Lib\site-packages\transformers-*.dist-info

!python -m pip install --no-cache-dir torch==2.5.1+cu121 torchvision==0.20.1+cu121 torchaudio==2.5.1+cu121 --index-url https://download.pytorch.org/whl/cu121
!python -m pip install --no-cache-dir transformers==4.51.3 torchao==0.10.0 tokenizers==0.21.1
!python -m pip install --no-cache-dir unsloth==2026.5.7 unsloth-zoo==2026.5.4 --no-deps
import torch
from unsloth import FastLanguageModel

max_seq_length = 1024
load_in_4bit = True

dtype = None
if torch.cuda.is_available():
    major, _ = torch.cuda.get_device_capability()
    dtype = torch.bfloat16 if major >= 8 else torch.float16
else:
    dtype = torch.float32

model_name = "Qwen/Qwen3-4B"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

print("Tokenizer pad token:", tokenizer.pad_token)
print("EOS token:", tokenizer.eos_token)


Found existing installation: unsloth 2026.5.7
Uninstalling unsloth-2026.5.7:
  Successfully uninstalled unsloth-2026.5.7
Found existing installation: unsloth_zoo 2026.5.4
Uninstalling unsloth_zoo-2026.5.4:
  Successfully uninstalled unsloth_zoo-2026.5.4
Found existing installation: transformers 4.57.6
Uninstalling transformers-4.57.6:
  Successfully uninstalled transformers-4.57.6
Found existing installation: torchao 0.17.0
Uninstalling torchao-0.17.0:
  Successfully uninstalled torchao-0.17.0
Found existing installation: tokenizers 0.22.2
Uninstalling tokenizers-0.22.2:
  Successfully uninstalled tokenizers-0.22.2
Files removed: 3563 (9514.9 MB)
Directories removed: 1664


The system cannot find the file specified.
The filename, directory name, or volume label syntax is incorrect.


Looking in indexes: https://download.pytorch.org/whl/cu121
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB ? eta -:--:--
     ---------------------------------------- 0.0/2.4 GB 8.4 MB/s eta 0:04:51
     ---------------------------------------- 0.0/2.4 GB 5.4 MB/s eta 0:07:32
     ---------------------------------------- 0.0/2.4 GB 7.8 MB/s eta 0:05:13
     ---------------------------------------- 0.0/2.4 GB 9.5 MB/s eta 0:04:19
     ---------------------------------------- 0.0/2.4 GB 9.7 MB/s eta 0:04:11
     ---------------------------------------- 0.0/2.4 GB 9.6 MB/s eta 0:04:15
     ---------------------------------------- 0.0/2.4 GB 9.6 MB/s eta 0:04:15
     ---------------------------------------- 0.0/2.4 GB 9.9 MB/s eta 0:04:06
     ---------------------------------------- 0.0/2.4 GB 9.8 MB/s eta 0:04:08
     ---------------------------------------- 0.0/2.4 GB 9.8 MB/s eta 0:04:09
     --------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bert-score 0.3.13 requires transformers>=3.0.0, which is not installed.
peft 0.19.1 requires transformers, which is not installed.
sentence-transformers 5.5.1 requires transformers<6.0.0,>=4.41.0, which is not installed.
trl 0.24.0 requires transformers>=4.56.1, which is not installed.
xformers 0.0.35 requires torch>=2.10, but you have torch 2.5.1+cu121 which is incompatible.


   ---------------------------------------- 0.0/10.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/10.4 MB ? eta -:--:--
   ------ --------------------------------- 1.6/10.4 MB 8.4 MB/s eta 0:00:02
   ------------ --------------------------- 3.1/10.4 MB 7.1 MB/s eta 0:00:02
   ------------------------ --------------- 6.3/10.4 MB 9.6 MB/s eta 0:00:01
   -------------------------------- ------- 8.4/10.4 MB 9.6 MB/s eta 0:00:01
   ---------------------------------------- 10.4/10.4 MB 10.3 MB/s  0:00:01
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------------------------ --------- 1.8/2.4 MB 9.1 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 9.9 MB/s  0:00:00
   ---------------------------------------- 0.0/746.6 kB ? eta -:--:--
   ---------------------------------------- 746.6/746.6 kB 7.8 MB/s  0:00:00

   ---------------------------------------- 0/3 [torchao]
   ---------------------------------------- 0/3 [to

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
trl 0.24.0 requires transformers>=4.56.1, but you have transformers 4.51.3 which is incompatible.


   ---------------------------------------- 0.0/71.1 MB ? eta -:--:--
   ---------------------------------------- 0.5/71.1 MB 3.4 MB/s eta 0:00:22
    --------------------------------------- 1.6/71.1 MB 7.6 MB/s eta 0:00:10
   -- ------------------------------------- 4.5/71.1 MB 8.7 MB/s eta 0:00:08
   --- ------------------------------------ 6.3/71.1 MB 8.8 MB/s eta 0:00:08
   ---- ----------------------------------- 8.7/71.1 MB 9.1 MB/s eta 0:00:07
   ------ --------------------------------- 10.7/71.1 MB 9.5 MB/s eta 0:00:07
   ------- -------------------------------- 13.1/71.1 MB 9.7 MB/s eta 0:00:06
   -------- ------------------------------- 15.2/71.1 MB 9.8 MB/s eta 0:00:06
   --------- ------------------------------ 17.3/71.1 MB 9.9 MB/s eta 0:00:06
   ----------- ---------------------------- 19.7/71.1 MB 10.1 MB/s eta 0:00:06
   ------------ --------------------------- 21.8/71.1 MB 10.0 MB/s eta 0:00:05
   ------------- -------------------------- 23.6/71.1 MB 9.9 MB/s eta 0:00:

c:\Users\hanib\AppData\Local\Programs\Python\Python310\lib\site-packages\unsloth\__init__.py:144: UserWarning: WARNING: Unsloth should be imported before [transformers] to ensure all optimizations are applied. Your code may run slower or encounter memory issues without these optimizations.

Please restructure your imports with 'import unsloth' at the top of your file.
  from ._gpu_init import *


Exception: cannot import name 'VideoInput' from 'transformers.image_utils' (c:\Users\hanib\AppData\Local\Programs\Python\Python310\lib\site-packages\transformers\image_utils.py)

In [ ]:

def apply_chat_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": text}

train_ds = datasets_dict["train"].map(apply_chat_template)
val_ds = datasets_dict["validation"].map(apply_chat_template)
test_ds = datasets_dict["test"].map(apply_chat_template)

print(train_ds[0]["text"][:500])


## Weights & Biases

In [ ]:

import wandb

WANDB_PROJECT = os.getenv("WANDB_PROJECT", "abstract-evaluator-qwen25-3b")
WANDB_RUN_NAME = os.getenv("WANDB_RUN_NAME", "qwen25-3b-unsloth-qlora")

wandb.login()
wandb.init(project=WANDB_PROJECT, name=WANDB_RUN_NAME)


## Train (SFTTrainer)

In [ ]:

from trl import SFTTrainer
from transformers import TrainingArguments

num_epochs = 3

training_args = TrainingArguments(
    output_dir=str(PROJECT_ROOT / "outputs" / "qwen25_3b_abstract_eval"),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,
    learning_rate=2e-4,
    weight_decay=0.01,
    num_train_epochs=num_epochs,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    fp16=(dtype == torch.float16),
    bf16=(dtype == torch.bfloat16),
    report_to=["wandb"],
    run_name=WANDB_RUN_NAME,
    gradient_checkpointing=True,
    dataloader_pin_memory=True,
    seed=SEED,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    packing=False,
    args=training_args,
)

train_result = trainer.train()
print(train_result)


In [ ]:

adapter_dir = PROJECT_ROOT / "outputs" / "qwen25_3b_abstract_eval" / "lora_adapter"
trainer.model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print("Saved adapter to:", adapter_dir)


## Validation + Test Generation and Metrics

In [ ]:

import evaluate
from tqdm.auto import tqdm

rouge_metric = evaluate.load("rouge")
bleu_metric = evaluate.load("bleu")
bertscore_metric = evaluate.load("bertscore")

FastLanguageModel.for_inference(model)


def parse_gold(messages):
    text = messages[1]["content"]
    try:
        obj = json.loads(text)
    except Exception:
        obj = {"score": None, "rationale": text}
    return obj


def parse_pred(text):
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if not match:
        return {"score": None, "rationale": text.strip()}
    chunk = match.group(0)
    try:
        return json.loads(chunk)
    except Exception:
        return {"score": None, "rationale": text.strip()}


def generate_predictions(ds, max_new_tokens=180):
    preds, refs = [], []
    pred_scores, ref_scores = [], []

    for row in tqdm(ds):
        msgs = row["messages"]
        prompt = tokenizer.apply_chat_template(
            [msgs[0]], tokenize=False, add_generation_prompt=True
        )
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                temperature=0.0,
                top_p=1.0,
                eos_token_id=tokenizer.eos_token_id,
            )
        gen_text = tokenizer.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        pred_obj = parse_pred(gen_text)
        gold_obj = parse_gold(msgs)

        preds.append(str(pred_obj.get("rationale", "")))
        refs.append(str(gold_obj.get("rationale", "")))
        pred_scores.append(pred_obj.get("score", None))
        ref_scores.append(gold_obj.get("score", None))

    return preds, refs, pred_scores, ref_scores


val_preds, val_refs, val_pred_scores, val_ref_scores = generate_predictions(datasets_dict["validation"])
test_preds, test_refs, test_pred_scores, test_ref_scores = generate_predictions(datasets_dict["test"])

val_rouge = rouge_metric.compute(predictions=val_preds, references=val_refs)
val_bleu = bleu_metric.compute(predictions=val_preds, references=[[r] for r in val_refs])
val_bertscore = bertscore_metric.compute(predictions=val_preds, references=val_refs, lang="en")

print("Validation ROUGE:", val_rouge)
print("Validation BLEU:", val_bleu)
print("Validation BERTScore F1 mean:", float(np.mean(val_bertscore["f1"])))


In [ ]:

test_rouge = rouge_metric.compute(predictions=test_preds, references=test_refs)
test_bleu = bleu_metric.compute(predictions=test_preds, references=[[r] for r in test_refs])
test_bertscore = bertscore_metric.compute(predictions=test_preds, references=test_refs, lang="en")

valid_pairs = [(p, r) for p, r in zip(test_pred_scores, test_ref_scores) if isinstance(p, int) and isinstance(r, int)]
score_acc = float(np.mean([int(p == r) for p, r in valid_pairs])) if valid_pairs else None

results = {
    "test_rouge": test_rouge,
    "test_bleu": test_bleu,
    "test_bertscore_f1_mean": float(np.mean(test_bertscore["f1"])),
    "test_score_accuracy": score_acc,
    "num_valid_score_pairs": len(valid_pairs),
}

print(json.dumps(results, indent=2))

wandb.log({
    "test/rouge1": test_rouge.get("rouge1", 0.0),
    "test/rouge2": test_rouge.get("rouge2", 0.0),
    "test/rougeL": test_rouge.get("rougeL", 0.0),
    "test/bleu": test_bleu.get("bleu", 0.0),
    "test/bertscore_f1_mean": float(np.mean(test_bertscore["f1"])),
    "test/score_accuracy": score_acc if score_acc is not None else 0.0,
})

results_path = PROJECT_ROOT / "outputs" / "qwen25_3b_abstract_eval" / "final_metrics.json"
results_path.parent.mkdir(parents=True, exist_ok=True)
with open(results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)
print("Saved:", results_path)



## Notes

- For 8GB VRAM, start with `max_seq_length=1024`, `batch_size=1`, `grad_accum=16`, QLoRA 4-bit.
- If OOM occurs:
  - reduce `max_seq_length` to `768` or `512`
  - increase `gradient_accumulation_steps` instead of batch size
  - set LoRA `r=8`
- If underfitting, try 4 epochs; if validation loss rises while train loss drops, keep 2-3 epochs.
